In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
diabetes = pd.read_csv('./data/raw/cdc_places.csv')
food = pd.read_csv('./data/raw/food_access.csv')

In [3]:
diabetes.head()

,Year,StateAbbr,StateDesc,CountyName,CountyFIPS,LocationName,DataSource,Category,Measure,Data_Value_Unit,...,Data_Value_Footnote,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,Geolocation,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text
0,2018,AL,Alabama,Baldwin,1003,1003011601,BRFSS,Prevention,Cervical cancer screening among adult women ag...,%,...,NaN,83.5,87.1,"6,062",POINT (-87.55879555 30.41466842),1003011601,PREVENT,CERVICAL,CrdPrv,Cervical Cancer Screening
1,2019,ME,Maine,Waldo,23027,23027045000,BRFSS,Health Status,Fair or poor self-rated health status among ad...,%,...,NaN,17.4,20.3,"5,969",POINT (-69.23270168 44.42005508),23027045000,HLTHSTAT,GHLTH,CrdPrv,General Health
2,2019,AL,Alabama,Calhoun,1015,1015002300,BRFSS,Health Outcomes,Obesity among adults aged >=18 years,%,...,NaN,38.0,40.0,"3,843",POINT (-85.67101785 33.93457433),1015002300,HLTHOUT,OBESITY,CrdPrv,Obesity
3,2019,LA,Louisiana,Tangipahoa,22105,22105953500,BRFSS,Health Risk Behaviors,Current smoking among adults aged >=18 years,%,...,NaN,22.3,25.4,"7,225",POINT (-90.36823496 30.77718215),22105953500,RISKBEH,CSMOKING,CrdPrv,Current Smoking
4,2019,ME,Maine,Cumberland,23005,23005014000,BRFSS,Health Risk Behaviors,Current smoking among adults aged >=18 years,%,...,NaN,14.7,20.3,"3,872",POINT (-70.61427542 43.97054122),23005014000,RISKBEH,CSMOKING,CrdPrv,Current Smoking


In [4]:
food.head()

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractSeniors,TractWhite,TractBlack,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP
0,1001020100,Alabama,Autauga County,1,1912,693,0,0.0,0.00,0,...,221.0,1622.0,217.0,14.0,0.0,14.0,45.0,44.0,6.0,102.0
1,1001020200,Alabama,Autauga County,1,2170,743,0,181.0,8.34,1,...,214.0,888.0,1217.0,5.0,0.0,5.0,55.0,75.0,89.0,156.0
2,1001020300,Alabama,Autauga County,1,3373,1256,0,0.0,0.00,0,...,439.0,2576.0,647.0,17.0,5.0,11.0,117.0,87.0,99.0,172.0
3,1001020400,Alabama,Autauga County,1,4386,1722,0,0.0,0.00,0,...,904.0,4086.0,193.0,18.0,4.0,11.0,74.0,85.0,21.0,98.0
4,1001020500,Alabama,Autauga County,1,10766,4082,0,181.0,1.68,0,...,1126.0,8666.0,1437.0,296.0,9.0,48.0,310.0,355.0,230.0,339.0


In [5]:
diabetes = diabetes[diabetes['Measure'].str.contains('diabetes')] # Filters to only diabetes cases
diabetes = diabetes[['LocationName', 'Data_Value']]
diabetes.head()

,LocationName,Data_Value
11,23025966600,12.2
13,23031034002,8.7
50,23025965600,12.8
67,1083020300,10.6
72,1087231900,18.1


In [6]:
# Select Features
food = food[['CensusTract', 'State', 'County', 'LATracts_half', 'LATracts1', 
             'LATracts10', 'LATracts20', 'Pop2010', 'Urban', 'PCTGQTRS', 'PovertyRate',
             'MedianFamilyIncome', 'TractSeniors', 'TractWhite', 'TractBlack',
             'TractAsian', 'TractNHOPI', 'TractAIAN', 'TractHispanic']]

In [7]:
# Turns raw counts into rates
food['TractSeniorsRate'] = food['TractSeniors'] / food['Pop2010']
food['TractWhiteRate'] = food['TractWhite'] / food['Pop2010']
food['TractBlackRate'] = food['TractBlack'] / food['Pop2010']
food['TractAsianRate'] = food['TractAsian'] / food['Pop2010']
food['TractNHOPIRate'] = food['TractNHOPI'] / food['Pop2010']
food['TractAIANRate'] = food['TractAIAN'] / food['Pop2010']
food['TractHispanicRate'] = food['TractHispanic'] / food['Pop2010']

# Turns percentages into proportions
food['PovertyRatePerc'] = food['PovertyRate'] / 100
food['GroupQuarters'] = food['PCTGQTRS'] / 100

# Drops raw count columns
food = food.drop(['TractSeniors', 'TractWhite', 'TractBlack', 'TractAsian', 'TractNHOPI', 'TractAIAN',
           'TractHispanic', 'PovertyRate', 'PCTGQTRS'], axis = 1) 

In [8]:
food.describe()

,CensusTract,LATracts_half,LATracts1,LATracts10,LATracts20,Pop2010,Urban,MedianFamilyIncome,TractSeniorsRate,TractWhiteRate,TractBlackRate,TractAsianRate,TractNHOPIRate,TractAIANRate,TractHispanicRate,PovertyRatePerc,GroupQuarters
count,7.253100e+04,72531.000000,72531.000000,72531.000000,72531.000000,72531.000000,72531.000000,71783.000000,72527.000000,72527.000000,72527.000000,72527.000000,72527.000000,72527.000000,72527.000000,72528.000000,72506.000000
mean,2.782573e+10,0.638830,0.335884,0.043926,0.004784,4256.739022,0.760626,77037.792249,0.136264,0.718700,0.138354,0.044031,0.001649,0.010171,0.152714,0.151839,0.027087
std,1.581647e+10,0.480343,0.472302,0.204932,0.069002,1955.987626,0.426704,37544.445885,0.073877,0.257003,0.222952,0.083982,0.010085,0.046881,0.208271,0.119199,0.095709
min,1.001020e+09,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,2499.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.212708e+10,0.000000,0.000000,0.000000,0.000000,2899.000000,1.000000,51484.000000,0.090757,0.582829,0.010968,0.004780,0.000000,0.001926,0.024174,0.065000,0.000000
50%,2.712979e+10,1.000000,0.000000,0.000000,0.000000,4011.000000,1.000000,68821.000000,0.129112,0.808100,0.039901,0.014722,0.000309,0.003859,0.062069,0.120000,0.001800
75%,4.103900e+10,1.000000,1.000000,0.000000,0.000000,5330.500000,1.000000,93868.500000,0.167260,0.921795,0.148322,0.043130,0.001063,0.007671,0.180889,0.206000,0.015700
max,5.604595e+10,1.000000,1.000000,1.000000,1.000000,37452.000000,1.000000,250001.000000,1.000000,1.000000,1.000000,1.000000,0.858824,1.000000,1.000000,1.000000,1.000000


In [9]:
# Turns diabetes percentage into a rate
diabetes['DiabetesRate'] = diabetes['Data_Value'] / 100
diabetes = diabetes.drop('Data_Value', axis = 1)

In [10]:
diabetes.describe()

,LocationName,DiabetesRate
count,7.033800e+04,70338.000000
mean,2.764672e+10,0.109630
std,1.600635e+10,0.037273
min,1.001020e+09,0.007000
25%,1.210501e+10,0.085000
50%,2.700305e+10,0.104000
75%,4.106703e+10,0.128000
max,5.604595e+10,0.442000


In [11]:
# Dataset shapes
print(diabetes.shape)
print(food.shape)

(70338, 2)
(72531, 19)


In [12]:
# Combines Datasets
df = diabetes.merge(food, left_on = 'LocationName', right_on = 'CensusTract', how = 'inner')

# Drop one tract ID column
df = df.drop('CensusTract', axis = 1)
df.head()

,LocationName,DiabetesRate,State,County,LATracts_half,LATracts1,LATracts10,LATracts20,Pop2010,Urban,MedianFamilyIncome,TractSeniorsRate,TractWhiteRate,TractBlackRate,TractAsianRate,TractNHOPIRate,TractAIANRate,TractHispanicRate,PovertyRatePerc,GroupQuarters
0,23025966600,0.122,Maine,Somerset County,1,1,0,0,4269,1,59453.0,0.206372,0.971656,0.003748,0.007964,0.000000,0.001640,0.004919,0.157,0.0054
1,23031034002,0.087,Maine,York County,0,0,0,0,5352,0,88786.0,0.129671,0.972720,0.003737,0.004858,0.000187,0.001308,0.013079,0.027,0.0002
2,23025965600,0.128,Maine,Somerset County,0,0,0,0,2028,0,42143.0,0.168639,0.968442,0.001479,0.003945,0.000000,0.006410,0.005424,0.192,0.0099
3,1083020300,0.106,Alabama,Limestone County,0,0,0,0,3583,0,63516.0,0.145967,0.967346,0.012280,0.000837,0.000000,0.004745,0.008094,0.081,0.0000
4,1087231900,0.181,Alabama,Macon County,1,1,0,0,1940,1,53654.0,0.132474,0.009278,0.972680,0.002577,0.000000,0.000515,0.008247,0.285,0.0000


In [13]:
# Create final food access feature
df.insert(2, 'FoodAccessScore', np.select(
    [
        df['LATracts20'] == 1,
        df['LATracts10'] == 1,
        df['LATracts1'] == 1,
        df['LATracts_half'] == 1
    ],
    [
        4, 
        3, 
        2, 
        1, 
    ], default = 0
))

# Drop original food access columns
df = df.drop(['LATracts_half', 'LATracts1', 'LATracts10', 'LATracts20'], axis = 1)
df.head()

,LocationName,DiabetesRate,FoodAccessScore,State,County,Pop2010,Urban,MedianFamilyIncome,TractSeniorsRate,TractWhiteRate,TractBlackRate,TractAsianRate,TractNHOPIRate,TractAIANRate,TractHispanicRate,PovertyRatePerc,GroupQuarters
0,23025966600,0.122,2,Maine,Somerset County,4269,1,59453.0,0.206372,0.971656,0.003748,0.007964,0.000000,0.001640,0.004919,0.157,0.0054
1,23031034002,0.087,0,Maine,York County,5352,0,88786.0,0.129671,0.972720,0.003737,0.004858,0.000187,0.001308,0.013079,0.027,0.0002
2,23025965600,0.128,0,Maine,Somerset County,2028,0,42143.0,0.168639,0.968442,0.001479,0.003945,0.000000,0.006410,0.005424,0.192,0.0099
3,1083020300,0.106,0,Alabama,Limestone County,3583,0,63516.0,0.145967,0.967346,0.012280,0.000837,0.000000,0.004745,0.008094,0.081,0.0000
4,1087231900,0.181,2,Alabama,Macon County,1940,1,53654.0,0.132474,0.009278,0.972680,0.002577,0.000000,0.000515,0.008247,0.285,0.0000


In [14]:
df.isnull().sum()

LocationName            0
DiabetesRate            0
FoodAccessScore         0
State                   0
County                  0
Pop2010                 0
Urban                   0
MedianFamilyIncome    545
TractSeniorsRate        4
TractWhiteRate          4
TractBlackRate          4
TractAsianRate          4
TractNHOPIRate          4
TractAIANRate           4
TractHispanicRate       4
PovertyRatePerc         3
GroupQuarters          24
dtype: int64

In [15]:
# Replace missing values with the median
df['MedianFamilyIncome'] = df['MedianFamilyIncome'].replace(np.nan, df['MedianFamilyIncome'].median())
df['TractSeniorsRate'] = df['TractSeniorsRate'].replace(np.nan, df['TractSeniorsRate'].median())
df['TractWhiteRate'] = df['TractWhiteRate'].replace(np.nan, df['TractWhiteRate'].median())
df['TractBlackRate'] = df['TractBlackRate'].replace(np.nan, df['TractBlackRate'].median())
df['TractAsianRate'] = df['TractAsianRate'].replace(np.nan, df['TractAsianRate'].median())
df['TractNHOPIRate'] = df['TractNHOPIRate'].replace(np.nan, df['TractNHOPIRate'].median())
df['TractAIANRate'] = df['TractAIANRate'].replace(np.nan, df['TractAIANRate'].median())
df['TractHispanicRate'] = df['TractHispanicRate'].replace(np.nan, df['TractHispanicRate'].median())
df['PovertyRatePerc'] = df['PovertyRatePerc'].replace(np.nan, df['PovertyRatePerc'].median())
df['GroupQuarters'] = df['GroupQuarters'].replace(np.nan, df['GroupQuarters'].median())

In [16]:
df.isnull().sum()

LocationName          0
DiabetesRate          0
FoodAccessScore       0
State                 0
County                0
Pop2010               0
Urban                 0
MedianFamilyIncome    0
TractSeniorsRate      0
TractWhiteRate        0
TractBlackRate        0
TractAsianRate        0
TractNHOPIRate        0
TractAIANRate         0
TractHispanicRate     0
PovertyRatePerc       0
GroupQuarters         0
dtype: int64

In [17]:
# tests for collinearity 
df[['FoodAccessScore', 'Urban', 'MedianFamilyIncome', 'TractSeniorsRate', 'PovertyRatePerc', 'GroupQuarters',
        'TractWhiteRate', 'TractBlackRate', 'TractAsianRate', 'TractNHOPIRate', 'TractAIANRate', 'TractHispanicRate']].corr()

,FoodAccessScore,Urban,MedianFamilyIncome,TractSeniorsRate,PovertyRatePerc,GroupQuarters,TractWhiteRate,TractBlackRate,TractAsianRate,TractNHOPIRate,TractAIANRate,TractHispanicRate
FoodAccessScore,1.000000,0.332340,0.065892,0.044753,-0.081761,0.020653,0.062699,-0.013213,-0.062956,0.000424,0.059155,-0.082686
Urban,0.332340,1.000000,0.096301,-0.148143,0.097005,0.028543,-0.316687,0.183332,0.234977,0.043219,-0.101422,0.233587
MedianFamilyIncome,0.065892,0.096301,1.000000,0.054405,-0.635942,-0.041655,0.316542,-0.325942,0.251571,-0.002225,-0.098125,-0.240357
TractSeniorsRate,0.044753,-0.148143,0.054405,1.000000,-0.191184,-0.123525,0.324628,-0.163554,-0.117007,-0.047742,-0.048144,-0.289912
PovertyRatePerc,-0.081761,0.097005,-0.635942,-0.191184,1.000000,0.170291,-0.473153,0.443280,-0.102064,-0.004369,0.113187,0.239478
GroupQuarters,0.020653,0.028543,-0.041655,-0.123525,0.170291,1.000000,-0.050847,0.073893,-0.000512,0.018260,0.001399,-0.032819
TractWhiteRate,0.062699,-0.316687,0.316542,0.324628,-0.473153,-0.050847,1.000000,-0.809352,-0.298622,-0.136071,-0.136890,-0.300851
TractBlackRate,-0.013213,0.183332,-0.325942,-0.163554,0.443280,0.073893,-0.809352,1.000000,-0.107537,-0.038965,-0.058962,-0.078426
TractAsianRate,-0.062956,0.234977,0.251571,-0.117007,-0.102064,-0.000512,-0.298622,-0.107537,1.000000,0.208245,-0.043801,0.079708
TractNHOPIRate,0.000424,0.043219,-0.002225,-0.047742,-0.004369,0.018260,-0.136071,-0.038965,0.208245,1.000000,0.000126,0.041158


In [18]:
# Additional collinearity test
X = df[['FoodAccessScore', 'Urban', 'MedianFamilyIncome', 'TractSeniorsRate', 'GroupQuarters', 'PovertyRatePerc',
        'TractWhiteRate', 'TractBlackRate', 'TractAsianRate', 'TractNHOPIRate', 'TractAIANRate', 'TractHispanicRate']]
y = df['DiabetesRate']

vif = pd.DataFrame()
vif['Variable'] = X.columns
vif['VIF'] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]
print(vif)

              Variable        VIF
0      FoodAccessScore   3.061355
1                Urban   6.160782
2   MedianFamilyIncome  10.449371
3     TractSeniorsRate   5.546377
4        GroupQuarters   1.144936
5      PovertyRatePerc   5.429972
6       TractWhiteRate  17.153132
7       TractBlackRate   3.361651
8       TractAsianRate   1.901548
9       TractNHOPIRate   1.080257
10       TractAIANRate   1.143485
11   TractHispanicRate   1.929148


In [19]:
# MedianFamilyIncome and white rate get dropped because of high VIF
X = df[['FoodAccessScore', 'Urban', 'TractSeniorsRate', 'GroupQuarters', 'PovertyRatePerc',
        'TractBlackRate', 'TractAsianRate', 'TractNHOPIRate', 'TractAIANRate', 'TractHispanicRate']]
y = df['DiabetesRate']

vif = pd.DataFrame()
vif['Variable'] = X.columns
vif['VIF'] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]
print(vif)

            Variable       VIF
0    FoodAccessScore  2.887471
1              Urban  5.322259
2   TractSeniorsRate  2.607502
3      GroupQuarters  1.129998
4    PovertyRatePerc  3.555459
5     TractBlackRate  1.977761
6     TractAsianRate  1.459175
7     TractNHOPIRate  1.076616
8      TractAIANRate  1.098565
9  TractHispanicRate  1.923749


In [20]:
# export csv
df.to_csv("./data/processed/diabetes_food_clean.csv", index=False)